# LangChain Tools

A **tool** is a callable function that a language model can request when it needs information or an action that is outside its built-in knowledge. LangChain turns ordinary Python functions into model-usable tools by attaching a name, description, and input schema to them.

This notebook demonstrates the complete tool-calling workflow:

1. Initialize a chat model.
2. Define and register a typed Python function with the `@tool` decorator.
3. Bind the tool to the model so the model knows it is available.
4. Inspect the model's tool call and its generated arguments.
5. Execute the tool and send its result back to the model for a final response.

The weather function returns mock data for learning purposes; it does not query a live weather service.

In [1]:
import torch

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"config\.env")


# --- ROCm/CUDA device check ---
# ROCm exposes itself to PyTorch through the same torch.cuda API as NVIDIA CUDA,
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detected: {device_name} ({total_vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected by torch — falling back to CPU. Check your ROCm/torch install.")


GPU detected: AMD Radeon RX 7900 XT (20.0 GB VRAM)


In [2]:
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

GRmodel = init_chat_model("groq:qwen/qwen3.6-27b",)
response = GRmodel.invoke("Why do parrots have such colorful feathers?")
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots have such colorful feathers?" This is a biological/evolutionary question about parrot plumage coloration.\n\n2.  **Identify Key Concepts**:\n   - Parrot feather colors (vibrant, diverse)\n   - Evolutionary reasons for coloration\n   - Biological mechanisms (pigments, structural colors)\n   - Functions: sexual selection, species recognition, camouflage, social signaling, health indicators\n   - Environmental/ecological context (tropical habitats, light conditions)\n\n3.  **Research/Recall Knowledge**:\n   - *Pigments*: Carotenoids (reds, yellows, oranges), porphyrins (reds, greens, blues - though rare in birds, psittacofulvins are unique to parrots), melanins (blacks, browns, grays)\n   - *Structural colors*: Iridophores, feather microstructure creating blues, greens, iridescence\n   - *Psittacofulvins*: Unique pigments to parrots, responsible for reds and yellows, synthesized from

### Model setup

`init_chat_model` creates a chat model using a provider-qualified model name. The `invoke` method sends one prompt and returns an `AIMessage`, which contains the model's response content and any metadata, such as requested tool calls.

In [8]:
from langchain.tools import tool

@tool #decorator to register the function as a tool
def get_weather(location: str) -> str:
    """
    Get the current weather for a given location.
    """
    # For demonstration purposes, we'll return a mock response.
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model = GRmodel.bind_tools([get_weather]) # Bind the tool to the model

### Defining and binding a tool

The `@tool` decorator registers `get_weather` as a LangChain tool. Its type annotation, `location: str`, becomes part of the input schema, while the docstring becomes the description supplied to the model.

`bind_tools` makes the tool available to the model. Binding does not execute the function; it only gives the model enough information to decide when to request it and which arguments to generate.

In [9]:
response = model.invoke("What is the weather like in New York City today?")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'New York City'}


### Inspecting a tool call

When the model decides to use a tool, it returns a structured **tool call** rather than running Python itself. Each tool call includes the registered tool name and an `args` object containing arguments that match the tool's input schema.

The next cell prints those fields so you can inspect the model's decision before executing anything.

## Tool Execution Loop

A tool call is only a request from the model, so the application must execute it and return the result. This creates a three-step loop:

1. **Model decision:** invoke the model and collect its structured tool calls.
2. **Tool execution:** call the matching Python function with the generated arguments, then append the tool result to the message history.
3. **Final response:** invoke the model again with the complete conversation so it can explain the result naturally.

`messages` is the conversation history. It contains the original user request, the model's tool-call message, and the tool result message. Keeping these messages in order gives the model the context it needs for the final answer.

In [10]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What is the weather like in New York City?"}]
ai_msg = model.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute the tool calls and collect the results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response generation
final_response = model.invoke(messages)
print(final_response.content)
# The final_response.content will contain the model's response after considering the tool's output.

The current weather in New York City is sunny with a temperature of 25°C.


In [11]:
messages

[{'role': 'user', 'content': 'What is the weather like in New York City?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to know the current weather in New York City.\n2.  **Identify Available Tools**: I have a tool `get_weather` that takes a `location` parameter.\n3.  **Extract Parameters**: Location = "New York City"\n4.  **Call Tool**: `get_weather(location="New York City")`\n5.  **Process Response**: Wait for the tool output, then formulate the final answer. (Self-correction/Refinement: I will just call the tool as requested.)✅\n', 'tool_calls': [{'id': '3pwn944zp', 'function': {'arguments': '{"location":"New York City"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 280, 'total_tokens': 434, 'completion_time': 0.30117, 'completion_tokens_details': {'reasoning_tokens': 124}, 'prompt_time': 0.021224281, 'prompt_tokens_deta